## **Lab 3: Spectral Representation**

The goal of this lab is to connect the time-domain signal you have been working with to its frequency-domain representation. The theory summary on spectrum representation introduces two complementary views of the same signal:

- the Fourier transform, which gives the frequency content of a signal segment, and
- the spectrogram, which shows how that content changes over time.

In this lab we will first compute the magnitude spectrum of a short fragment of the reference audio, then we will compute a spectrogram of the whole sound. The goal is not only to produce plots, but also to interpret them and relate them back to the sound you hear.

A periodic signal contains energy at its fundamental frequency and at its harmonics. When the signal is not perfectly periodic, the spectrum becomes less sharp and the energy spreads around those frequencies. In a short fragment, the FFT gives a snapshot of the dominant components. In a longer signal, the spectrogram shows how these components evolve over time.

In [1]:
import numpy as np
import librosa
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Audio, display

from util import load_audio, plot_signals, plot_spectrogram, plot_spectrum

We will use the same reference sound from the previous labs. If you have your own audio file in a cloned repository, you can point the path there; otherwise the notebook will use the bundled reference file.

### 1. Spectrum Analysis

This section show how to compute the spectrum and the spectrogram of a sound

In [ ]:
# Clone your own repository and point reference_path to your own audio file.
# !git clone https://github.com/your-username/sis1_name.git
# reference_path = Path('./sis1_name/your_file.wav')

reference_path = Path('./sis1_roger/157438__nengisuls__chunky-bass.wav')
reference_audio, fs = load_audio(reference_path)
print(f'Using audio file: {reference_path}')

display(Audio(reference_audio, rate=fs))

#### 1.1 Fourier transform of a short fragment

The first step is to look at a short fragment of the sound and compute its magnitude spectrum using the FFT. This gives a snapshot of the strongest frequency components present in that fragment.

In [ ]:
# Stable sustain region (after the pluck attack), ~0.18–0.28 s
start = int(0.18 * fs)
end = int(0.28 * fs)
fragment = reference_audio[start:end]

ax = plot_spectrum(fragment, sr=fs)
ax.set_ylim([-120, 0])
ax.set_xlim([0, 2000])  # focus on the first harmonics (~100 Hz spacing)

plt.show()

#### 1.2 Questions

1. What are the strongest peaks in the magnitude spectrum? To what harmonics do they correspond?
   - The strongest peak is near **200 Hz** (2nd harmonic). Next are ~**300 Hz** (3rd) and ~**100 Hz** (fundamental), then ~400 Hz (4th). Higher peaks are much weaker.

2. Do you see a fundamental frequency and its harmonics? How did you identify them?
   - Yes. Peaks sit at roughly **k · 100 Hz** (100, 200, 300, 400…). That spacing matches the Lab 1 `f0 ≈ 100 Hz` measured from the period.

3. How would the spectrum change if the fragment were taken from a different part of the signal?
   - Near the **attack**, higher harmonics are stronger/noisier (brighter transient). Later in the **decay**, overall level drops and higher harmonics fade faster, so the spectrum looks thinner and quieter.

#### 1.3 Spectrogram of the whole sound

The FFT of one fragment gives a local snapshot, but a sound changes over time. A spectrogram shows how the frequency content evolves across the whole signal.

In [ ]:
n_fft = 2048
hop_length = 256

ax = plot_spectrogram(reference_audio, sr=fs, N=n_fft, H=hop_length)
ax.set_ylim([0, 4000])
plt.show()

#### 1.4 Questions

1. Which frequencies are present for most of the sound? What does it mean from a perceptual point of view?
   - Low harmonics around **100–400 Hz** (especially ~200 Hz) stay visible for most of the note. That matches a low bass pitch that remains recognizable while it fades.

2. Do the bright bands stay fixed or do they change over time? What does it mean?
   - The band **frequencies** stay roughly fixed (stable pitch), but their **brightness** decays over time — amplitude envelopes change, not the note name.

3. How does the spectrogram help explain why the sound changes over time? Describe the sound evolution.
   - At the start there is a bright attack with more high-frequency energy; then the energy concentrates on the lower harmonics and slowly fades. That is why the pluck starts clicky/bright and becomes softer and darker.

#### 1.4 Fragment comparison

Repeat the spectral analysis for a different fragment of the signal and compare the magnitude spectra. Explain the difference.

In [ ]:
# Compare attack vs sustain magnitude spectra
frag_attack = reference_audio[int(0.02 * fs):int(0.08 * fs)]
frag_sustain = reference_audio[int(0.18 * fs):int(0.28 * fs)]

ax1 = plot_spectrum(frag_attack, sr=fs)
ax1.set_title('Attack fragment (0.02–0.08 s)')
ax1.set_ylim([-120, 0])
ax1.set_xlim([0, 2000])
plt.show()

ax2 = plot_spectrum(frag_sustain, sr=fs)
ax2.set_title('Sustain fragment (0.18–0.28 s)')
ax2.set_ylim([-120, 0])
ax2.set_xlim([0, 2000])
plt.show()

# Attack has stronger higher harmonics / less clean peak structure;
# sustain shows clearer peaks at k*100 Hz with H2 dominant.

Choose a time where several harmonics are visible in the spectrogram and plot the spectrum at that time. Can you explain the result?

In [ ]:
# Spectrum at a time where several harmonics are clearly visible (early sustain)
t_spec = 0.20
N = 4096
start_spec = int(t_spec * fs)
segment = reference_audio[start_spec:start_spec + N]

ax = plot_spectrum(segment, sr=fs)
ax.set_title(f'Spectrum starting at t = {t_spec:.2f} s (N={N})')
ax.set_ylim([-120, 0])
ax.set_xlim([0, 2000])
plt.show()

# Result: clear harmonic peaks at ~100, 200, 300, 400 Hz (and weaker higher ones),
# matching a pitched bass note with a strong 2nd harmonic.

Use the plots to estimate the weights of the fundamental frequency and a few harmonics. Store them in a list for later use. Be careful because the amplitude of the plot are in dB and you will need them in a linear scale, from 0 to 1, where 1 should be the maximum value of the spectrum. So you need to perform the conversion from dB to linear.

In [ ]:
# Measured from sustain spectrum (~0.18–0.28 s) at k*100 Hz peaks.
# Magnitudes were converted with A_lin = 10**(dB/20), then normalized so max = 1.
# H1..H8 linear magnitudes (relative): ~[0.19, 1.0, 0.43, 0.17, 0.022, 0.009, 0.010, 0.020]

f0_hz = np.array([100, 200, 300, 400, 500, 600, 700, 800], dtype=float)
# Example dB levels read from the spectrum peaks (relative to the strongest peak at 0 dB):
dB_rel = np.array([-14.4, 0.0, -7.4, -15.5, -33.1, -40.8, -39.8, -34.0])
lin = 10 ** (dB_rel / 20.0)
weights = (lin / lin.max()).tolist()

print('Harmonic frequencies [Hz]:', f0_hz.tolist())
print('Linear weights (max=1):', [round(w, 3) for w in weights])

### 2. Synthesis

Now we will use the harmonic synthesis idea from previous labs to build a signal from the estimated spectral weights.

In [ ]:
def synthesize(f0, phi, Ak, t):
    y = np.zeros_like(t, dtype=float)
    for k in range(1, len(Ak) + 1):
        y += Ak[k - 1] * np.cos(2 * np.pi * k * f0 * t + k * phi - (k - 1) * np.pi / 2)
    return y

#### 2.1 Write the code

Use the `synthesize` function to generate a signal with the weights from part 1.3 and the fundamental frequency and phase from earlier labs. Plot the synthesized signal together with the reference so you can compare them directly.

In [ ]:
# Reuse Lab 1 fundamental frequency and phase
f0 = 100.0
phi = -2.02

t = np.arange(len(reference_audio)) / fs
synth = synthesize(f0, phi, weights, t)

# Match overall amplitude to the reference
synth = synth / np.max(np.abs(synth)) * np.max(np.abs(reference_audio))

print('Synthesized with', len(weights), 'harmonics')
display(Audio(synth, rate=fs))

In [ ]:
# Time-domain comparison on a few stable periods (same window as Labs 1–2)
period = 1 / f0
t_start = 0.18
t_end = t_start + 8 * period

plot_signals(
    [reference_audio, synth],
    fs,
    t_start=t_start,
    t_end=t_end,
    name=['reference', 'harmonic synthesis'],
)

# Spectrogram comparison
ax1 = plot_spectrogram(reference_audio, sr=fs, N=2048, H=256)
ax1.set_ylim([0, 2000])
ax1.set_title('Reference spectrogram')
plt.show()

ax2 = plot_spectrogram(synth, sr=fs, N=2048, H=256)
ax2.set_ylim([0, 2000])
ax2.set_title('Synthesized spectrogram (constant harmonic weights)')
plt.show()

#### 2.2 Questions

1. What do you notice when you compare the synthesized spectrogram with the reference spectrogram?
   - The synthesis shows the same harmonic frequencies (lines at ~100, 200, 300… Hz), but they stay **constant in brightness** for the whole duration. The reference lines **fade** and the attack is brighter/noisier.

2. Which parts of the sound are reproduced well, and which parts are still missing or different?
   - **Well reproduced:** pitch / harmonic spacing and the relative strength of the first few partials in a stable window (esp. strong H2).
   - **Missing/different:** the pluck attack transient, the decaying amplitude envelope, time-varying harmonic balance, and any noise/inharmonic energy.